In [5]:
import pandas as pd
import numpy as np 
import random
from tmdbv3api import Movie, Person, TMDb
import requests


path_to_data = '../data/the_oscar_award.csv'

cat_to_role_dict = {
        'actor':'actor', 
        'actress': 'actress',
        'directing': 'director',
        'writing':'writer',
    }


In [6]:
def big5_Oscar_df(path):
    '''
    Function reads in a csv containing data on oscar nominees and converts into a pandas dataframe
    
    Param: path of csv file
    Returns: pandas dataframe
    
    '''

    #Take in oscar nominee csv
    oscar_df = pd.read_csv(path)

    #select for all columns that contain strings
    str_cols = oscar_df.select_dtypes('str').columns
    #convert all rows of each column with str type to lowercase
    oscar_df[str_cols] = oscar_df[str_cols].apply(lambda x: x.str.lower())
    # handle cases of multiple nominees, giving each their own row
    oscar_df['name'] = oscar_df['name'].astype(str)
    oscar_df['name'] = oscar_df['name'].str.split('/')
    oscar_df = oscar_df.explode('name')
    # pre-process role column to be usable later
    role_dict = {
        'actor':'actor', 
        'actress': 'actress',
        'director': 'director',
        'writer':'writer',
    }
    
    for key, value in role_dict.items():
        oscar_df.loc[oscar_df['role'].str.contains(key), 'role'] = value

    # limit df to only pull from big five categories
    oscar_df = oscar_df[oscar_df['role'].isin(role_dict.keys()) | (oscar_df['category'] == 'best picture')]
        
    

    return oscar_df





In [7]:
class category: 

    def __init__(self, df, category):
        
        self.category = category
        self.df = df[df['category'].str.contains(self.category)]
        self.movie_list = list(self.df['film'])


In [8]:
class nominee: 

    def __init__(self, name, df, cat, role):

        self.name = name
        self.role = role
        if cat_to_role_dict[cat] == self.role:
            self.df = df[(df['role'] == role) & (df['name'] == self.name)]
        else: 
            self.df = df[df['film'].isin(self._get_filmography())]
        self.c = category(df, cat)
        # Isolate only the oscar movies the nominee has appeared in for this category (even if not the awards intended recipient)
        self.nom_category_appearance = self.df[
            self.df['film'].isin(self.c.df['film']) & 
            self.df['category'].isin(self.c.df['category'])
        ]
        

    def _get_filmography(self):

        tmdb = TMDb()
        person = Person()
        movie = Movie()

        tmdb.api_key = 'f49d2ebf0d11031312ade120f61c513d'

        nominee_search = person.search(self.name)

        nomID = nominee_search[0].id

        url = f"https://api.themoviedb.org/3/person/{nomID}/combined_credits"
        params = {"api_key": tmdb.api_key}
        
        data = requests.get(url, params=params).json()

        movies = data['cast']

        filmography = [m["title"] for m in movies if "title" in m]

        filmography = [f.lower() for f in filmography]

        return filmography
    
        

        

    def oscars_score(self):

        '''
        function calculates a weight based on oscar nominations and wins. Wins are heavily favored, but nominations are not counted against as they would be in a probabilistic determination 
        of success 
        
        param: self <- object of nominee class
        returns: weighted oscar score
        
        '''
        # Isolate only the winners from the category nominations the nominee has appeared in 
        outcomes = list(self.nom_category_appearance['winner'])

        # initialize oscar score
        oscar_score = 0

        # If nominee has no previous nominations for category
        if len(outcomes) == 0:
            return 0
        else:
            # Iterate through each movie's outcome and add weighted score to oscar score
            for decision in outcomes:
                if cat_to_role_dict[self.c.category] == self.role:
                    if decision == True: 
                    #arbitrary weighting, favorable weighting for a win
                        oscar_score += 5
                    else: 
                        oscar_score += 3
                elif cat_to_role_dict[self.c.category] != self.role:
                    if decision == True:
                        oscar_score += 1
                    else: 
                        oscar_score += 0.5
                else: 
                    #arbitrary weighting, less favorable but still positive weighting for a nomination
                    oscar_score += 0

        return oscar_score

    def synergy_boost(self, crew_list):

        '''
        Function considers past professional relationships with other nominees and calculates a synergy boost.

        param1: self <- object of nominee class
        param2: crew_list <- list of other crew members on movie, each item in list is an instance of the nominee class

        returns: new score with added synergy boost
        '''
        
        # Initialize empy list for common movies
        all_common_movies = []

        # Initialize synergy score
        synergy_score = 0 

        # Iterate through crew list
        for member in crew_list:
            # Exclude self
            if member.name != self.name:
                # Takes the intersection between self and other crew members filmoraphy
                common_movies = set(member._get_filmography()) & set(self._get_filmography())
                # if 1 or more movies in common
                if len(common_movies) > 0:
                    # iterate through common movies
                    for movies in common_movies: 
                        # add to list of common movies shared between self and all other nominees 
                        all_common_movies.append(movies)        
                else:
                    # If no movies in common return a synergy score of 0
                    return 0
            else: 
                continue
                
        # iterate through list of all shared movies
        for movie in all_common_movies: 
            # If the movie won an oscar, add weight
            if movie in self.df[self.df['film'] == movie] & (self.df['winner'] == True):
                synergy_score += 0.6
            # If the movie was nominated but didn't win, add weight
            elif movie in self.df[self.df['film'] == movie] & (self.df['winner'] == False):
                synergy_score += 0.2
            # If movie wasn't nominated for some reason, pass to next movie
            else: 
                pass

        return synergy_score
    
    def __repr__(self):
        
        return self.name

In [9]:
oscar_df = big5_Oscar_df(path_to_data)

init_talent_pool = (oscar_df.groupby('role')['name'].apply(list).to_dict())

talent_pool = {}
for k, v in init_talent_pool.items(): 
    talent_pool[k] = list(set(v))

rng = np.random.default_rng(seed=56)


In [ ]:
class rand_movie: 

    def __init__(self, category, talent_pool, df):

        self.category = category

        self.talent_pool = talent_pool

    #def get_name():



    def hire_crew(self, role):

        i = rng.integers(low = 0,  high = int(len(talent_pool[role])))

        return nominee(talent_pool[role][i], oscar_df, self.category, role)
    


    def credits(self):

        

        self.actor = self.hire_crew('actor')

        self.actress = self.hire_crew('actress')

        self.director = self.hire_crew('director')

        self.writer = self.hire_crew('writer')

        self.crew_dict = {
            'actor': self.actor, 
            'actress': self.actress,
            'director': self.director,
            'writer': self.writer, 
        }

        return self.crew_dict
    
    def movie_oscar_score(self): 
        
        score = 0 

        for crew in self.crew_dict.values():  
            score += crew.oscars_score() + crew.synergy_boost(list(self.crew_dict.values()))
    
        return score  
    
    def __repr__(self): 

        return self.crew_dict
    

In [23]:
def metropolis_hastings(category, talent_pool, df, cat_to_role_dict, iterations = 1, conv_counter = 10, conv_threshhold = 0.5):

    c = 0 
    i = 0
    init_movie = rand_movie(category, talent_pool, oscar_df)
    init_crew = init_movie.credits()
    init_score = init_movie.movie_oscar_score()
    conv_check = np.inf

    while i <= iterations and c < conv_counter:

        print(f'{i} iterations')
        nom_movie = rand_movie(category, talent_pool, oscar_df)
        nom_crew = nom_movie.credits()
        nom_score = nom_movie.movie_oscar_score()

        scores = [init_score, nom_score]
        movies = [init_movie, nom_movie]
        
        movies

        init_movie = random.choices(movies, scores)

        if iterations % 2 == 0:
            conv_check = scores[0] - scores[1]
            if conv_check <= conv_threshhold:
                c += 1
            else: 
                c = 0
        i += 1
    
    return f'nominated for {category} is {init_crew[cat_to_role_dict[category]]} with an oscar score of {scores[0]}'
        



        




In [25]:
category_list = [x for x in cat_to_role_dict.keys() if oscar_df['category'].str.contains(x).any()]

for cat in category_list: 
    winners = metropolis_hastings(cat, talent_pool, oscar_df, cat_to_role_dict)
    print(winners)

0 iterations
1 iterations
nominated for actor is peter ustinov with an oscar score of 13.5
0 iterations
1 iterations
nominated for actress is glynis johns with an oscar score of 6.5
0 iterations


TypeError: attribute name must be string, not 'int'